<a href="https://colab.research.google.com/github/apirakqqqqq/GE337_Programming/blob/main/Lab_4/Lab_4_6606614870_%E0%B8%AD%E0%B8%A0%E0%B8%B4%E0%B8%A3%E0%B8%B1%E0%B8%81%E0%B8%A9%E0%B9%8C_%E0%B8%9B%E0%B8%B1%E0%B8%8D%E0%B8%8D%E0%B8%B2%E0%B8%AA%E0%B8%B2%E0%B8%84%E0%B8%A3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install geopandas rasterio folium shapely matplotlib -q

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


**โหลดข้อมูล**

In [4]:
import rasterio

raster_path = "/content/S2_BKK_7Bands.tif"

src = rasterio.open(raster_path)

print("CRS:", src.crs)
print("Width:", src.width)
print("Height:", src.height)
print("Number of bands:", src.count)
print("Bounds:", src.bounds)
print("Transform:", src.transform)

CRS: EPSG:4326
Width: 3340
Height: 3898
Number of bands: 7
Bounds: BoundingBox(left=100.29995574391057, bottom=13.399989430154077, right=100.90003035370242, top=14.100316025653656)
Transform: | 0.00, 0.00, 100.30|
| 0.00,-0.00, 14.10|
| 0.00, 0.00, 1.00|


In [5]:
import geopandas as gpd

vector_path = "/content/พื้นที่สีเขียว_เขตเมือง_แหล่งน้ำ_กรุงเทพ.json"

gdf = gpd.read_file(vector_path)

print(gdf.head())
print(gdf.columns)
print(gdf.crs)

   OBJECTID             F_id                                            name  \
0         1  node/3568027505                                 ็ต้นไม้รูปหัวใจ   
1         2  node/3752927040                    สนามกีฬา อำเภอพระสมุทรเจดีย์   
2         3  node/5260120841                                            None   
3         4  node/5382326737  Prachanukun Health Garden สวนสุขภาพ ประชานุกูล   
4         5  node/5614482723                                 Angoon's Garden   

  landuse natural water F_geometry                    geometry  
0    None    None  None       None   POINT (100.48982 13.7527)  
1    None    None  None       None  POINT (100.56669 13.56809)  
2    None    None  None       None  POINT (100.61142 13.63363)  
3    None    None  None       None  POINT (100.53834 13.83016)  
4    None    None  None       None   POINT (100.58034 13.7273)  
Index(['OBJECTID', 'F_id', 'name', 'landuse', 'natural', 'water', 'F_geometry',
       'geometry'],
      dtype='object')
EPSG:43

**คำนวณ NDVI**

In [6]:
import numpy as np

blue = src.read(1)
green = src.read(2)
red = src.read(3)
nir = src.read(4)
swir1 = src.read(5)
swir2 = src.read(6)

ndvi = (nir - red) / (nir + red + 1e-10)

**รวม Raster + Vector เป็น DataFrame**

In [7]:
coords = [(x,y) for x,y in zip(gdf.geometry.x, gdf.geometry.y)]

samples = []

for val in src.sample(coords):
    samples.append(val)

import pandas as pd

features = pd.DataFrame(samples, columns=[
    "Blue","Green","Red","NIR","SWIR1","SWIR2","Band7"
])

features["NDVI"] = (features["NIR"] - features["Red"]) / (features["NIR"] + features["Red"] + 1e-10)

df = pd.concat([gdf.reset_index(drop=True), features], axis=1)

print(df.head())

   OBJECTID             F_id                                            name  \
0         1  node/3568027505                                 ็ต้นไม้รูปหัวใจ   
1         2  node/3752927040                    สนามกีฬา อำเภอพระสมุทรเจดีย์   
2         3  node/5260120841                                            None   
3         4  node/5382326737  Prachanukun Health Garden สวนสุขภาพ ประชานุกูล   
4         5  node/5614482723                                 Angoon's Garden   

  landuse natural water F_geometry                    geometry    Blue  \
0    None    None  None       None   POINT (100.48982 13.7527)  1534.0   
1    None    None  None       None  POINT (100.56669 13.56809)   756.0   
2    None    None  None       None  POINT (100.61142 13.63363)   704.0   
3    None    None  None       None  POINT (100.53834 13.83016)   338.5   
4    None    None  None       None   POINT (100.58034 13.7273)   453.0   

    Green     Red     NIR   SWIR1   SWIR2     Band7      NDVI  
0  1790.0 

**แบ่ง Training / Testing**

In [10]:
df["class"] = "unknown"

df.loc[df["landuse"].notna(), "class"] = "urban"
df.loc[df["natural"] == "water", "class"] = "water"
df.loc[df["water"].notna(), "class"] = "water"

In [11]:
from sklearn.model_selection import train_test_split

X = df[["Blue","Green","Red","NIR","SWIR1","SWIR2","NDVI"]]
y = df["class"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

**ปรับขนาดข้อมูล + Feature Selection**

In [12]:
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

selector = SelectKBest(score_func=f_classif, k=5)

X_train_selected = selector.fit_transform(X_train_scaled, y_train)
X_test_selected = selector.transform(X_test_scaled)

**ใช้ Random Forest จำแนกพื้นที่**

In [13]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(random_state=42)

rf.fit(X_train_selected, y_train)

RandomForestClassifier(random_state=42)

**ใช้ Grid Search ปรับพารามิเตอร์**

In [14]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "n_estimators":[100,200],
    "max_depth":[5,10,None],
    "min_samples_split":[2,5]
}

grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring="accuracy"
)

grid.fit(X_train_selected, y_train)

best_model = grid.best_estimator_

print("Best Parameters:", grid.best_params_)

Best Parameters: {'max_depth': 10, 'min_samples_split': 5, 'n_estimators': 100}


In [15]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

y_pred = best_model.predict(X_test_selected)

print("Accuracy:", accuracy_score(y_test, y_pred))

print("Confusion Matrix")
print(confusion_matrix(y_test, y_pred))

print("Classification Report")
print(classification_report(y_test, y_pred))

Accuracy: 0.8346456692913385
Confusion Matrix
[[ 0  4  3]
 [ 0 55  5]
 [ 0  9 51]]
Classification Report
              precision    recall  f1-score   support

     unknown       0.00      0.00      0.00         7
       urban       0.81      0.92      0.86        60
       water       0.86      0.85      0.86        60

    accuracy                           0.83       127
   macro avg       0.56      0.59      0.57       127
weighted avg       0.79      0.83      0.81       127



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
